Duck Hunt — Cheat Detection: AI/ML Pillar

This is the third pillar of a three-pillar Streambased booth demo, all running against the same Kafka stream with no separate state job. Pillar 1 is Grafana on the live Kafka stream — operational, sub-second. Pillar 2 is a notebook on Iceberg coldset tables — analytical, arbitrary SQL after the fact. This notebook is pillar 3: an LLM scores each just-completed game against the full historical population, treating the whole game as a sequence rather than as isolated shots. **No stateful pre-aggregation, no separate state job — we read the whole game and the whole population on demand from Iceberg.** The structural advantage over Flink: you do not have to name your aggregations before the data arrives. The LLM gets the raw event tables and decides what matters.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from openai import OpenAI
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
# We're reading every event from a sampled subset of historical sessions as raw rows.
# With Flink we'd have had to pre-aggregate per session into a state store,
# naming the aggregates ahead of time. Here we hand the LLM the actual data.

import random

POPULATION_SAMPLE_FRACTION = 1.0  # fraction of historical sessions to load as context
POPULATION_SAMPLE_SEED = 42

_all_ids = [r['sessionId'] for r in spark.sql("SELECT DISTINCT sessionId FROM coldset.shots").collect()]
random.seed(POPULATION_SAMPLE_SEED)
_k = max(1, int(round(len(_all_ids) * POPULATION_SAMPLE_FRACTION)))
_sample_ids = random.sample(_all_ids, _k)
_in_clause = ",".join(f"'{sid}'" for sid in _sample_ids)
print(f"Sampling {_k} of {len(_all_ids)} sessions ({POPULATION_SAMPLE_FRACTION:.0%}) for population context")

pop_shots = spark.sql(f"""
SELECT
  sessionId,
  timestamp - MIN(timestamp) OVER (PARTITION BY sessionId) AS t_offset_ms,
  state, x, y
FROM coldset.shots
WHERE sessionId IN ({_in_clause})
ORDER BY sessionId, t_offset_ms
""").toPandas()

pop_spawns = spark.sql(f"""
SELECT
  sessionId,
  timestamp - MIN(timestamp) OVER (PARTITION BY sessionId) AS t_offset_ms,
  duck_id, x, y, direction
FROM coldset.duck_spawns
WHERE sessionId IN ({_in_clause})
ORDER BY sessionId, t_offset_ms
""").toPandas()

pop_gun = spark.sql(f"""
SELECT
  sessionId,
  timestamp - MIN(timestamp) OVER (PARTITION BY sessionId) AS t_offset_ms,
  x, y
FROM coldset.gun_positions
WHERE sessionId IN ({_in_clause})
ORDER BY sessionId, t_offset_ms
""").toPandas()

# Map sessionId -> compact session_idx in pandas (avoids unpartitioned window in SQL)
_session_ids = sorted(set(pop_shots['sessionId']) | set(pop_spawns['sessionId']) | set(pop_gun['sessionId']))
_id_to_idx = {sid: i for i, sid in enumerate(_session_ids)}
for _df in (pop_shots, pop_spawns, pop_gun):
    _df['session_idx'] = _df['sessionId'].map(_id_to_idx)
    _df.drop(columns=['sessionId'], inplace=True)

# Reorder columns so session_idx comes first
pop_shots  = pop_shots[['session_idx', 't_offset_ms', 'state', 'x', 'y']]
pop_spawns = pop_spawns[['session_idx', 't_offset_ms', 'duck_id', 'x', 'y', 'direction']]
pop_gun    = pop_gun[['session_idx', 't_offset_ms', 'x', 'y']]

print(f"Population: {pop_shots['session_idx'].nunique()} sessions, "
      f"{len(pop_shots)} shots, {len(pop_spawns)} spawns, {len(pop_gun)} gun positions")


In [ ]:
# Helper functions + POPULATION metrics. Run ONCE after loading the population (cell 3).
# These are static for the session — no need to recompute when scoring a new game.

def reaction_times_per_session(shots_df, spawns_df):
    results = {}
    for idx, s_grp in shots_df.groupby('session_idx'):
        sp_grp = spawns_df[spawns_df['session_idx'] == idx]
        rts = []
        for _, shot in s_grp.iterrows():
            prior = sp_grp[sp_grp['t_offset_ms'] <= shot['t_offset_ms']]
            if not prior.empty:
                rts.append(shot['t_offset_ms'] - prior['t_offset_ms'].max())
        if rts:
            results[idx] = float(np.median(rts))
    return results

def inter_shot_std_per_session(shots_df):
    results = {}
    for idx, grp in shots_df.groupby('session_idx'):
        diffs = grp['t_offset_ms'].sort_values().diff().dropna()
        if len(diffs) >= 2:
            results[idx] = float(diffs.std())
    return results

def cursor_jerk_per_session(gun_df):
    results = {}
    for idx, grp in gun_df.groupby('session_idx'):
        grp = grp.sort_values('t_offset_ms').reset_index(drop=True)
        dx = grp['x'].diff()
        dy = grp['y'].diff()
        dt = grp['t_offset_ms'].diff().clip(lower=1)
        speed = np.sqrt(dx**2 + dy**2) / dt
        jerk = speed.diff().abs()
        if jerk.dropna().shape[0] >= 2:
            results[idx] = float(jerk.std())
    return results

def time_on_target_per_session(shots_df, gun_df, radius=50):
    """ToT per shot: time elapsed since cursor was LAST OUTSIDE the radius around the shot.
    Captures dwell time on target before firing, not just 'time since last brush past'."""
    results = {}
    for idx, s_grp in shots_df.groupby('session_idx'):
        g_grp = gun_df[gun_df['session_idx'] == idx].sort_values('t_offset_ms')
        if g_grp.empty:
            continue
        tots = []
        for _, shot in s_grp.iterrows():
            before = g_grp[g_grp['t_offset_ms'] <= shot['t_offset_ms']]
            if before.empty:
                continue
            dists = np.sqrt((before['x'] - shot['x'])**2 + (before['y'] - shot['y'])**2)
            inside_radius = (dists <= radius)
            if not inside_radius.any():
                continue
            outside_radius = ~inside_radius
            if outside_radius.any():
                latest_outside_t = before.loc[outside_radius, 't_offset_ms'].max()
                tot = shot['t_offset_ms'] - latest_outside_t
            else:
                tot = shot['t_offset_ms'] - before['t_offset_ms'].min()
            tots.append(tot)
        if tots:
            results[idx] = float(np.median(tots))
    return results

def preaim_dist_per_session(gun_df, spawns_df, lookahead_ms=100):
    results = {}
    for idx, sp_grp in spawns_df.groupby('session_idx'):
        g_grp = gun_df[gun_df['session_idx'] == idx].sort_values('t_offset_ms')
        dists = []
        for _, spawn in sp_grp.iterrows():
            t_look = spawn['t_offset_ms'] - lookahead_ms
            before = g_grp[g_grp['t_offset_ms'] <= t_look]
            if not before.empty:
                last = before.iloc[-1]
                dists.append(np.sqrt((last['x'] - spawn['x'])**2 + (last['y'] - spawn['y'])**2))
        if dists:
            results[idx] = float(np.median(dists))
    return results

def accuracy_vs_speed_per_session(shots_df, spawns_df):
    results = {}
    for idx, s_grp in shots_df.groupby('session_idx'):
        sp_grp = spawns_df[spawns_df['session_idx'] == idx]
        hit_rates = []
        duck_times = []
        for _, shot in s_grp.iterrows():
            prior_spawns = sp_grp[sp_grp['t_offset_ms'] <= shot['t_offset_ms']]
            if not prior_spawns.empty:
                duck_times.append(shot['t_offset_ms'] - prior_spawns['t_offset_ms'].max())
            hit_rates.append(1 if shot['state'] == 'hit' else 0)
        if hit_rates:
            results[idx] = (float(np.mean(hit_rates)),
                            float(np.mean(duck_times)) if duck_times else float('nan'))
    return results

def _summarize(vals):
    arr = np.array([v for v in vals if not np.isnan(v)])
    if len(arr) == 0:
        return None
    return {
        'n':      int(len(arr)),
        'min':    float(arr.min()),
        'p5':     float(np.percentile(arr, 5)),
        'p10':    float(np.percentile(arr, 10)),
        'p25':    float(np.percentile(arr, 25)),
        'median': float(np.median(arr)),
        'p75':    float(np.percentile(arr, 75)),
        'p90':    float(np.percentile(arr, 90)),
        'p95':    float(np.percentile(arr, 95)),
        'max':    float(arr.max()),
        'mean':   float(arr.mean()),
        'std':    float(arr.std()),
    }

print('Computing population metrics (one-time)...')
pop_rt   = reaction_times_per_session(pop_shots, pop_spawns)
pop_isi  = inter_shot_std_per_session(pop_shots)
pop_jerk = cursor_jerk_per_session(pop_gun)
pop_tot  = time_on_target_per_session(pop_shots, pop_gun)
pop_pa   = preaim_dist_per_session(pop_gun, pop_spawns)
pop_avs  = accuracy_vs_speed_per_session(pop_shots, pop_spawns)
pop_acc_only = [v[0] for v in pop_avs.values()]

pop_summary = {
    'reaction_time_ms':         _summarize(list(pop_rt.values())),
    'inter_shot_interval_std':  _summarize(list(pop_isi.values())),
    'cursor_jerk':              _summarize(list(pop_jerk.values())),
    'time_on_target_ms':        _summarize(list(pop_tot.values())),
    'preaim_distance_px':       _summarize(list(pop_pa.values())),
    'accuracy':                 _summarize(pop_acc_only),
}
print('Population summary:')
for k, s in pop_summary.items():
    if s is None:
        print(f'  {k:30s} <no data>')
    else:
        print(f"  {k:30s} n={s['n']} median={s['median']:.2f}  p5={s['p5']:.2f}  p95={s['p95']:.2f}  mean={s['mean']:.2f}  σ={s['std']:.2f}")


In [ ]:
latest_row = spark.sql("""
SELECT sessionId
FROM merged.control_events
WHERE event_type = 'game_stop'
ORDER BY timestamp DESC
LIMIT 1
""").toPandas()

current_session_id = latest_row.iloc[0]['sessionId']
print(f"Latest completed session: {current_session_id}")

In [ ]:
# Same query shape as cell 3 but against `merged` — ISK serves the just-completed
# game's events from the live Kafka stream and the historical aggregates from Iceberg
# through one query interface. No state to rebuild, no aggregates to pre-name,
# no separate streaming job. The LLM gets the whole game as a sequence.

cur_shots = spark.sql(f"""
SELECT timestamp, state, x, y
FROM merged.shots
WHERE sessionId = '{current_session_id}'
ORDER BY timestamp
""").toPandas()

cur_spawns = spark.sql(f"""
SELECT timestamp, duck_id, x, y, direction
FROM merged.duck_spawns
WHERE sessionId = '{current_session_id}'
ORDER BY timestamp
""").toPandas()

cur_gun = spark.sql(f"""
SELECT timestamp, x, y
FROM merged.gun_positions
WHERE sessionId = '{current_session_id}'
ORDER BY timestamp
""").toPandas()

# Compute t_offset_ms in pandas (avoids unpartitioned SQL window)
_t0 = min(
    cur_shots['timestamp'].min()  if len(cur_shots)  else np.inf,
    cur_spawns['timestamp'].min() if len(cur_spawns) else np.inf,
    cur_gun['timestamp'].min()    if len(cur_gun)    else np.inf,
)
for _df in (cur_shots, cur_spawns, cur_gun):
    _df['t_offset_ms'] = _df['timestamp'] - _t0
    _df['session_idx'] = 0
    _df.drop(columns=['timestamp'], inplace=True)

cur_shots  = cur_shots[['session_idx', 't_offset_ms', 'state', 'x', 'y']]
cur_spawns = cur_spawns[['session_idx', 't_offset_ms', 'duck_id', 'x', 'y', 'direction']]
cur_gun    = cur_gun[['session_idx', 't_offset_ms', 'x', 'y']]

print(f"Current game: {len(cur_shots)} shots, {len(cur_spawns)} spawns, {len(cur_gun)} gun positions")

In [ ]:
# CURRENT-GAME metrics. Re-run this whenever you load a new session (cells 4+5).
# Uses helper functions defined in the population-metrics cell.

cur_rt   = next(iter(reaction_times_per_session(cur_shots, cur_spawns).values()), float('nan'))
cur_isi  = next(iter(inter_shot_std_per_session(cur_shots).values()), float('nan'))
cur_jerk = next(iter(cursor_jerk_per_session(cur_gun).values()), float('nan'))
cur_tot  = next(iter(time_on_target_per_session(cur_shots, cur_gun).values()), float('nan'))
cur_pa   = next(iter(preaim_dist_per_session(cur_gun, cur_spawns).values()), float('nan'))
cur_avs  = next(iter(accuracy_vs_speed_per_session(cur_shots, cur_spawns).values()),
                (float('nan'), float('nan')))

metrics_summary = {
    'reaction_time_ms':         {'population': pop_summary['reaction_time_ms'],         'current_game': cur_rt},
    'inter_shot_interval_std':  {'population': pop_summary['inter_shot_interval_std'],  'current_game': cur_isi},
    'cursor_jerk':              {'population': pop_summary['cursor_jerk'],              'current_game': cur_jerk},
    'time_on_target_ms':        {'population': pop_summary['time_on_target_ms'],        'current_game': cur_tot},
    'preaim_distance_px':       {'population': pop_summary['preaim_distance_px'],       'current_game': cur_pa},
    'accuracy':                 {'population': pop_summary['accuracy'],                 'current_game': cur_avs[0]},
}
print('Current game:')
for k, v in metrics_summary.items():
    pop = v['population']
    if pop is None:
        print(f"  {k:30s} current={v['current_game']:.2f}  pop=<no data>")
    else:
        print(f"  {k:30s} current={v['current_game']:.2f}  pop median={pop['median']:.2f} σ={pop['std']:.2f} p5={pop['p5']:.2f} p95={pop['p95']:.2f}")


In [ ]:
# Toggle: True = send percentile-ladder population stats (default — distributional shape).
# False = send raw event CSVs (LLM derives everything; heavier, needs strong model).
USE_DISTRIBUTIONAL_STATS = True

def _fmt_dist(metric_name, summary):
    if summary is None:
        return f'{metric_name}: <no data>'
    return (f"{metric_name}: n={summary['n']}  "
            f"min={summary['min']:.2f}  p5={summary['p5']:.2f}  p10={summary['p10']:.2f}  "
            f"p25={summary['p25']:.2f}  median={summary['median']:.2f}  p75={summary['p75']:.2f}  "
            f"p90={summary['p90']:.2f}  p95={summary['p95']:.2f}  max={summary['max']:.2f}  "
            f"mean={summary['mean']:.2f}  std={summary['std']:.2f}")

pop_stats_block = '\n'.join([
    _fmt_dist('reaction_time_ms',        metrics_summary['reaction_time_ms']['population']),
    _fmt_dist('inter_shot_interval_std', metrics_summary['inter_shot_interval_std']['population']),
    _fmt_dist('cursor_jerk',             metrics_summary['cursor_jerk']['population']),
    _fmt_dist('time_on_target_ms',       metrics_summary['time_on_target_ms']['population']),
    _fmt_dist('preaim_distance_px',      metrics_summary['preaim_distance_px']['population']),
    _fmt_dist('accuracy',                metrics_summary['accuracy']['population']),
])

cur_stats_block = '\n'.join([
    f"reaction_time_ms:        {metrics_summary['reaction_time_ms']['current_game']:.2f}",
    f"inter_shot_interval_std: {metrics_summary['inter_shot_interval_std']['current_game']:.2f}",
    f"cursor_jerk:             {metrics_summary['cursor_jerk']['current_game']:.4f}",
    f"time_on_target_ms:       {metrics_summary['time_on_target_ms']['current_game']:.2f}",
    f"preaim_distance_px:      {metrics_summary['preaim_distance_px']['current_game']:.2f}",
    f"accuracy:                {metrics_summary['accuracy']['current_game']:.3f}",
])

# Build raw CSV strings (used only for raw mode)
pop_shots_csv  = pop_shots.to_csv(index=False)
pop_spawns_csv = pop_spawns.to_csv(index=False)
_full_gun_csv  = pop_gun.to_csv(index=False)
_total_pre_gun = len(pop_shots_csv) + len(pop_spawns_csv)
_gun_budget    = max(100_000, 700_000 - _total_pre_gun)
if len(_full_gun_csv) > _gun_budget:
    _step = max(1, int(np.ceil(len(pop_gun) / max(1, _gun_budget // max(1, len(_full_gun_csv) // max(1, len(pop_gun)) + 1)))))
    pop_gun_sampled = pop_gun.iloc[::_step].reset_index(drop=True)
else:
    pop_gun_sampled = pop_gun
pop_gun_csv    = pop_gun_sampled.to_csv(index=False)
cur_shots_csv  = cur_shots.to_csv(index=False)
cur_spawns_csv = cur_spawns.to_csv(index=False)
cur_gun_csv    = cur_gun.to_csv(index=False)

_pop_n = metrics_summary['reaction_time_ms']['population']['n'] if metrics_summary['reaction_time_ms']['population'] else 'n/a'

system_prompt = """\
You are a cheat-detection analyst for a duck-hunt arcade game. You are an analytical reasoner,
not a rule-applier. Your job is to look at a just-completed game's metric values alongside the
DISTRIBUTIONAL SHAPE of the historical clean-player population (full percentile ladder per
metric) and reason about whether this game's pattern of values is consistent with clean play
or with cheating.

WHAT YOU KNOW:
- The historical population consists entirely of CLEAN players with a wide range of skill
  styles. Within-clean variance is large by design — fast/sloppy players have short
  time-on-target AND lower accuracy; careful/sniper players have long time-on-target AND
  higher accuracy. A single metric being at one end of the population distribution is NOT
  evidence of cheating on its own.
- Cheating produces patterns that no clean skill profile can produce — combinations of
  metric values that are mutually inconsistent with normal human play.
- The known cheat archetypes are: aimbot, scripted, triggerbot, preaim. We do NOT give you
  numeric definitions for these — you must reason from first principles about what each one
  would mechanically do to the metrics, and whether the observed pattern matches.

CORE PRINCIPLE — cheats produce IMPROVEMENT, not degradation:
  Cheating exists to give the player superhuman advantages: faster reflexes, perfect aim,
  knowledge of future events, robotic timing. By definition, a player who is *worse* than the
  population (slower reactions, lower accuracy, more erratic timing, jerkier cursor, longer
  hesitation on target) cannot be cheating — they have nothing to gain. A cheat verdict requires
  that the observed pattern represents an UNNATURAL ADVANTAGE: performance that exceeds human
  physical or cognitive limits, OR behaviour that humans cannot reproduce (e.g. cursor
  anticipating future spawns, robotic fixed-cadence timing).
  
  When a metric is outside the population's range, ask: "is this person doing BETTER than
  humans can, or WORSE?" If worse, it's a bad/unusual clean player and not relevant to cheating.

HOW TO REASON:
1. For each metric, locate the current game's value on the population's percentile ladder.
   Is it inside the population's normal range, near a tail, or beyond any percentile shown?
2. Consider the JOINT pattern across metrics. Are the deviations mutually consistent with
   any clean skill style (e.g. \"this looks like a careful sniper\" or \"this looks like a
   sloppy spray-and-pray player\")?
3. If you cannot construct a plausible clean-player story that explains the joint pattern,
   consider whether any cheat archetype would mechanically produce it. Cheating typically
   gives you superhuman timing, superhuman accuracy, suspiciously regular timing, or cursor
   behaviour that anticipates events. Reason about which one would mechanically produce the
   observed deviations.
4. Be especially cautious of declaring suspicion on a SINGLE outlier metric. Clean players
   span the full population. Two or more mutually-coherent deviations are needed for high
   confidence cheating.

METRIC DIRECTIONALITY — only ONE side of each metric is a cheat indicator:
  - reaction_time_ms: LOW values (below population p5) suggest aimbot or preaim.
    HIGH values are just slow/careful humans — NOT a cheat signal.
  - accuracy: HIGH values (above population p95) suggest aimbot.
    LOW values are just unskilled humans — NOT a cheat signal.
  - time_on_target_ms: LOW values (near zero, well below p5) suggest triggerbot.
    HIGH values are just careful humans dwelling on target — NOT a cheat signal.
  - inter_shot_interval_std: LOW values (below p5) suggest scripted/macro.
    HIGH values are just erratic timing — NOT a cheat signal.
  - preaim_distance_px: LOW values (below p5) suggest preaim/wallhack.
    HIGH values are just poor anticipation — NOT a cheat signal.
  - cursor_jerk: LOW values may suggest aimbot smoothness (weak signal).
    HIGH values are just shaky/jittery cursor — NOT a cheat signal.
  If a metric is outside the population's range in the WRONG direction (the "not a cheat signal"
  direction), do not cite it as evidence of cheating. It only means the player is unusually
  slow/inaccurate/careful/etc.

OUTPUT:
- verdict: clean if you can explain the pattern with normal human variance OR if your
  confidence in a cheat verdict is below 0.6. Suspicious only when you would assign
  confidence >= 0.6 to a specific cheat archetype.
- archetype: clean | aimbot | scripted | triggerbot | preaim — your best inference.
- confidence: how strongly the pattern coheres at the chosen verdict. >=0.85 only when
  multiple metrics together point at one story. 0.6-0.85 when one signal is striking but
  others are normal. If you would otherwise assign <0.6 confidence to a cheat archetype,
  return verdict="clean" instead — a single ambiguous outlier is not evidence of cheating.
- cited_features: 1-3 metric names that drove your reasoning. Use names from this list only:
  reaction_time, inter_shot_interval, cursor_jerk, accuracy_vs_speed, time_on_target,
  preaim_distance.
- reasoning: ONE concise sentence (~180 chars) describing your read of the joint pattern
  and which story it fits, citing specific values from the data.

You must call the submit_verdict tool. Do not return free text.
"""

user_message_dist = f"""\
## HISTORICAL CLEAN-PLAYER POPULATION — per-metric distributional shape (sessions={_pop_n})
Each line: min, p5/p10/p25/median/p75/p90/p95, max, mean, std across population sessions.

{pop_stats_block}

## JUST-COMPLETED GAME — single-session metric values
{cur_stats_block}

Reason about whether the current game's joint pattern of values is consistent with the
clean population's distributional shape, or whether some combination of deviations cannot
be explained by clean skill variance.
"""

user_message_raw = f"""\
## RAW EVENT TABLES — no pre-computed metrics. Derive what you need.

### Historical population — shots
{pop_shots_csv}

### Historical population — duck_spawns
{pop_spawns_csv}

### Historical population — gun_positions (sampled)
{pop_gun_csv}

### Just-completed game — shots
{cur_shots_csv}

### Just-completed game — duck_spawns
{cur_spawns_csv}

### Just-completed game — gun_positions
{cur_gun_csv}
"""

user_message = user_message_dist if USE_DISTRIBUTIONAL_STATS else user_message_raw

print(f"Prompt mode: {'distributional stats' if USE_DISTRIBUTIONAL_STATS else 'raw events'}  |  "
      f"user_message size: {len(user_message):,} chars")


In [ ]:
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    print("OPENAI_API_KEY not set — skipping LLM verdict, using placeholder")
    verdict_dict = {
        "verdict": "clean",
        "archetype": "clean",
        "confidence": 0.0,
        "reasoning": "LLM not configured",
        "cited_features": []
    }
else:
    client = OpenAI(api_key=api_key)
    schema = {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "verdict": {"type": "string", "enum": ["clean", "suspicious"]},
            "archetype": {"type": "string", "enum": ["clean", "aimbot", "scripted", "triggerbot", "preaim"]},
            "confidence": {"type": "number"},
            "reasoning": {"type": "string"},
            "cited_features": {
                "type": "array",
                "items": {"type": "string", "enum": ["reaction_time", "inter_shot_interval", "cursor_jerk", "accuracy_vs_speed", "time_on_target", "preaim_distance"]}
            }
        },
        "required": ["verdict", "archetype", "confidence", "reasoning", "cited_features"]
    }
    response = client.chat.completions.create(
        model="o4-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "submit_verdict",
                "strict": True,
                "schema": schema
            }
        }
    )
    verdict_dict = json.loads(response.choices[0].message.content)

# POST-PROCESS guard: low-confidence cheat verdicts get downgraded to clean.
if verdict_dict.get("verdict") == "suspicious" and verdict_dict.get("confidence", 0) < 0.6:
    verdict_dict["verdict"] = "clean"
    verdict_dict["archetype"] = "clean"
    verdict_dict["reasoning"] = (
        f"[downgraded from suspicious; confidence {verdict_dict.get('confidence', 0):.2f} below 0.6 threshold] "
        + verdict_dict.get("reasoning", "")
    )

print(verdict_dict)

In [ ]:
# Metrics already computed in the metrics-summary cell above (cell-5b).
# Nothing to do here — kept as a placeholder to preserve cell numbering.
print('Metrics already computed in earlier cell.')


In [ ]:
border_color = "#d62728" if verdict_dict['verdict'] == 'suspicious' else "#2ca02c"

# Map cited_features strings to panel indices (0-based, row-major in 2x3 grid)
_cited = set(verdict_dict.get('cited_features', []))
_feature_to_panel = {
    'reaction_time':       0,
    'inter_shot_interval': 1,
    'cursor_jerk':         2,
    'accuracy_vs_speed':   3,
    'time_on_target':      4,
    'preaim_distance':     5,
}
_highlighted_panels = {_feature_to_panel[f] for f in _cited if f in _feature_to_panel}

fig, axes = plt.subplots(2, 3, figsize=(17, 9))

for i, ax in enumerate(axes.flat):
    if i in _highlighted_panels:
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(3.5)

def _hist_kde_panel(ax, pop_vals, cur_val, title, xlabel, panel_idx):
    vals = np.array([v for v in pop_vals if not np.isnan(v)])
    if len(vals) > 1:
        ax.hist(vals, bins=20, density=True, alpha=0.45, color='steelblue', label='population')
        from scipy.stats import gaussian_kde
        kde = gaussian_kde(vals, bw_method='scott')
        xs = np.linspace(vals.min() * 0.8, vals.max() * 1.2, 300)
        ax.plot(xs, kde(xs), color='steelblue', linewidth=1.5)
    if not np.isnan(cur_val):
        ax.axvline(cur_val, color=border_color, linewidth=2.5, label=f'this game: {cur_val:.1f}')
        # Ensure cur_val is visible even if it's outside the population range
        if len(vals) > 0:
            lo = min(vals.min(), cur_val) * 0.9 if min(vals.min(), cur_val) >= 0 else min(vals.min(), cur_val) * 1.1
            hi = max(vals.max(), cur_val) * 1.1
            ax.set_xlim(lo, hi)
    weight = 'bold' if panel_idx in _highlighted_panels else 'normal'
    ax.set_title(title, fontsize=11, fontweight=weight)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('density', fontsize=10)
    ax.legend(fontsize=9)

_hist_kde_panel(axes[0,0], list(pop_rt.values()),   cur_rt,   'Reaction Time (median ms/session)', 'ms', 0)
_hist_kde_panel(axes[0,1], list(pop_isi.values()),  cur_isi,  'Inter-Shot Interval σ (ms)',        'ms', 1)
_hist_kde_panel(axes[0,2], list(pop_jerk.values()), cur_jerk, 'Cursor Jerk (σ of Δspeed)',         'px/ms²', 2)

# Panel 3 — Accuracy vs duck speed (scatter, not histogram)
ax = axes[1,0]
pop_acc  = [v[0] for v in pop_avs.values() if not np.isnan(v[0])]
pop_dspd = [v[1] for v in pop_avs.values() if not np.isnan(v[1])]
if pop_acc and pop_dspd:
    ax.scatter(pop_dspd, pop_acc, alpha=0.4, color='steelblue', s=20, label='population')
if not np.isnan(cur_avs[0]):
    ax.scatter([cur_avs[1]], [cur_avs[0]], color=border_color, s=160, zorder=5, marker='*',
               label=f'this game: acc={cur_avs[0]:.2f}')
    # Make sure the current-game marker is inside the visible area
    if pop_dspd and pop_acc:
        x_lo, x_hi = min(min(pop_dspd), cur_avs[1]) * 0.9, max(max(pop_dspd), cur_avs[1]) * 1.1
        y_lo = min(min(pop_acc), cur_avs[0]) - 0.05
        y_hi = max(max(pop_acc), cur_avs[0]) + 0.05
        ax.set_xlim(x_lo, x_hi)
        ax.set_ylim(max(0, y_lo), min(1.05, y_hi))
weight = 'bold' if 3 in _highlighted_panels else 'normal'
ax.set_title('Accuracy vs Duck Time on Screen', fontsize=11, fontweight=weight)
ax.set_xlabel('median ms duck alive before shot', fontsize=10)
ax.set_ylabel('hit rate', fontsize=10)
ax.legend(fontsize=9)
if 3 in _highlighted_panels:
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3.5)

_hist_kde_panel(axes[1,1], list(pop_tot.values()), cur_tot, 'Time on Target (median ms)',                  'ms', 4)
_hist_kde_panel(axes[1,2], list(pop_pa.values()),  cur_pa,  'Pre-aim Distance (median px at t-100ms)',     'px', 5)

# Verdict banner — split header (bold/colored) and reasoning (regular/black)
archetype  = verdict_dict['archetype']
confidence = verdict_dict['confidence']
reasoning  = verdict_dict['reasoning']
header = (
    f"Session: {current_session_id[:8]}…  |  "
    f"Verdict: {verdict_dict['verdict'].upper()}  |  "
    f"Archetype: {archetype}  |  "
    f"Confidence: {confidence:.0%}"
)
fig.text(0.5, 0.99, header, ha='center', va='top',
         fontsize=11, fontweight='bold', color=border_color)
fig.text(0.5, 0.955, reasoning, ha='center', va='top',
         fontsize=10, fontweight='normal', color='black', wrap=True)

# Outer border driven by verdict
rect = mpatches.FancyBboxPatch(
    (0.005, 0.005), 0.99, 0.99,
    boxstyle="square,pad=0",
    linewidth=10, edgecolor=border_color, facecolor='none',
    transform=fig.transFigure, clip_on=False
)
fig.add_artist(rect)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()


In [ ]:
# Track the last flagged session across runs.
if verdict_dict['verdict'] == 'suspicious':
    last_flagged_session_id = current_session_id

try:
    print(f"Last flagged session: {last_flagged_session_id}")
    print("To review it: set current_session_id = last_flagged_session_id, then re-run cells 5–9.")
    # Uncomment to jump straight to last flagged and re-run from cell 5:
    # current_session_id = last_flagged_session_id
except NameError:
    print("No session flagged as suspicious yet in this notebook run.")